# External Kaggle Financial Risk Evaluation

**AI Business Risk — reproducible experiment notebook**

This notebook reproduces/summarizes the external-data research experiment on the two
independent Kaggle datasets:

1. `data/raw/kaggle/bankruptcy/Bankruptcy.csv` (5,000 x 65, **unlabeled test matrix**)
2. `data/raw/kaggle/financial statements/Financial Statements.csv` (161 x 23, 12 companies, 2009-2023)

Rules honored by this notebook:

- Raw CSVs are read-only; SHA-256 verified against recorded provenance.
- Datasets are evaluated **independently** — never merged.
- Frozen Phase 5 Isolation Forest / rule baseline run **unmodified** (seed 42).
- Phase 4 Risk Engine called **unmodified**; missing inputs stay `MISSING_INPUT`.
- **No label-gated metrics are computed** — neither dataset has genuine labels.
- All outputs below are generated by executing this notebook (no static fake outputs).

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "backend").exists():  # allow launching from the notebook folder
    REPO = Path.cwd().parents[1]
sys.path.insert(0, str(REPO))

from backend.external_data.bankruptcy import (  # noqa: E402
    feature_frame,
    load_bankruptcy,
    target_columns,
)
from backend.external_data.contracts import RANDOM_SEED  # noqa: E402
from backend.external_data.evaluation import run_isolation_forest  # noqa: E402
from backend.external_data.financial_statements import (  # noqa: E402
    load_financial_statements,
    panel_summary,
)
from backend.external_data.preprocessing import build_feature_matrix, leakage_audit  # noqa: E402
from backend.external_data.provenance import sha256_file  # noqa: E402
from backend.external_data.risk_mapping import (  # noqa: E402
    availability_summary,
    evaluate_companies,
)
from backend.ml_engine.baseline import rolling_rule_baseline  # noqa: E402

RAW_BK = REPO / "data" / "raw" / "kaggle" / "bankruptcy" / "Bankruptcy.csv"
RAW_FS = REPO / "data" / "raw" / "kaggle" / "financial statements" / "Financial Statements.csv"
PROCESSED = REPO / "data" / "processed" / "kaggle"
FIGURES = REPO / "docs" / "assets" / "external-evaluation"

def read_json(path: Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

print(f"repo root : {REPO}")
print(f"seed      : {RANDOM_SEED}")
print(f"bankruptcy: {RAW_BK.exists()}")
print(f"financials: {RAW_FS.exists()}")

repo root : E:\AI Business Risk
seed      : 42
bankruptcy: True
financials: True


## 1. Provenance / hash verification (live)

In [2]:
# Live SHA-256 of both raw files, compared against recorded provenance.
bk_hash = sha256_file(RAW_BK)
fs_hash = sha256_file(RAW_FS)
bk_prov = read_json(PROCESSED / "bankruptcy" / "provenance.json")
fs_prov = read_json(PROCESSED / "financial-statements" / "provenance.json")

verification = pd.DataFrame(
    [
        {"dataset": "bankruptcy", "sha256_now": bk_hash, "sha256_recorded": bk_prov["sha256"]},
        {
            "dataset": "financial-statements",
            "sha256_now": fs_hash,
            "sha256_recorded": fs_prov["sha256"],
        },
    ]
)
verification["match"] = verification["sha256_now"] == verification["sha256_recorded"]
print(verification.to_string(index=False))
assert verification["match"].all(), "raw file hash mismatch — STOP"
print("\nBoth raw files match their recorded provenance (unchanged by the experiment run).")

             dataset                                                       sha256_now                                                  sha256_recorded  match
          bankruptcy 4d3c07357112cd3839d3c813363743777be032f2899604ba01219a9936eeba64 4d3c07357112cd3839d3c813363743777be032f2899604ba01219a9936eeba64   True
financial-statements 32e38512643a626340b43a4255bf79d45a7afc66ffe9058326507cc2885cb1b5 32e38512643a626340b43a4255bf79d45a7afc66ffe9058326507cc2885cb1b5   True

Both raw files match their recorded provenance (unchanged by the experiment run).


## 2. Dataset inspection (live)

**Bankruptcy label status first** — the label policy requires verifying whether a genuine
target exists before any evaluation is attempted.

In [3]:
bk_raw = load_bankruptcy(RAW_BK)
bk_features = feature_frame(bk_raw)
print("BANKRUPTCY")
print("  shape:", bk_raw.shape, "| feature matrix:", bk_features.shape)
print("  first columns:", list(bk_raw.columns[:3]), "...", list(bk_raw.columns[-1:]))
print("  duplicate rows:", int(bk_raw.duplicated().sum()))
print("  target columns present:", target_columns(bk_raw) or "NONE")
print("  => supervised labels UNAVAILABLE -> unsupervised-only evaluation (automatic fallback)")

BANKRUPTCY
  shape: (5000, 65) | feature matrix: (5000, 64)
  first columns: ['ID', 'Attr1', 'Attr2'] ... ['Attr64']
  duplicate rows: 0
  target columns present: NONE
  => supervised labels UNAVAILABLE -> unsupervised-only evaluation (automatic fallback)


In [4]:
fs_raw = load_financial_statements(RAW_FS)
fs_panel = panel_summary(fs_raw)
print("FINANCIAL STATEMENTS")
print("  shape:", fs_raw.shape)
print("  companies:", fs_panel["companies"], "| frequency:", fs_panel["frequency"])
print("  years:", fs_panel["years"][0], "-", fs_panel["years"][-1])
print(
    "  obs/company min-max:",
    fs_panel["min_obs_per_company"],
    "-",
    fs_panel["max_obs_per_company"],
)
missing = fs_raw.isna().sum()
missing = missing[missing > 0]
print("  missing cells per column:")
print(missing.to_string() if len(missing) else "    none")
print("  duplicate rows:", int(fs_raw.duplicated().sum()))
print("  duplicate company-year rows:", int(fs_raw.duplicated(subset=["Company", "Year"]).sum()))
print("  target columns present: NONE")

FINANCIAL STATEMENTS
  shape: (161, 23)
  companies: 12 | frequency: annual
  years: 2009 - 2023
  obs/company min-max: 9 - 15
  missing cells per column:
Market Cap(in B USD)    1


  duplicate rows: 0
  duplicate company-year rows: 0
  target columns present: NONE


## 3. Preprocessing + leakage audit (live, deterministic, seed 42)

In [5]:
FS_NUMERIC = [
    "Market Cap(in B USD)", "Revenue", "Gross Profit", "Net Income", "Earning Per Share",
    "EBITDA", "Share Holder Equity", "Cash Flow from Operating", "Cash Flow from Investing",
    "Cash Flow from Financial Activities", "Current Ratio", "Debt/Equity Ratio", "ROE", "ROA",
    "ROI", "Net Profit Margin", "Free Cash Flow per Share", "Return on Tangible Equity",
    "Number of Employees", "Inflation Rate(in US)",
]
fs_features = build_feature_matrix(
    fs_raw, [c for c in FS_NUMERIC if c in fs_raw.columns], dataset_id="financial-statements"
)

for dataset_id, matrix, target_cols in (
    ("bankruptcy", bk_features, target_columns(bk_raw)),
    ("financial-statements", fs_features, []),
):
    report = leakage_audit(
        dataset_id=dataset_id,
        feature_columns=list(matrix.columns),
        target_columns=target_cols,
        forbidden_columns=["ID", "Company", "Category", "Year"],
        preprocessing_uses_target=False,
        duplicate_cross_split=None,
        time_order_leakage=None,
    )
    passed = sum(c.passed for c in report.checks)
    print(f"{dataset_id}: leakage clean = {report.clean} ({passed}/{len(report.checks)} checks)")
    assert report.clean

print(
    "\nfeatures used — bankruptcy:",
    bk_features.shape[1],
    "| financial-statements:",
    fs_features.shape[1],
)
print("excluded: ID / Company / Year / Category (identifiers, time, metadata) — never features")

bankruptcy: leakage clean = True (5/5 checks)
financial-statements: leakage clean = True (5/5 checks)

features used — bankruptcy: 64 | financial-statements: 20
excluded: ID / Company / Year / Category (identifiers, time, metadata) — never features


## 4. Anomaly evaluation — frozen Phase 5 Isolation Forest + rule baseline (unmodified)

No tuning: `n_estimators=200`, `max_samples='auto'`, `max_features=1.0`, `bootstrap=False`,
robust median/IQR scaling, `random_state=42`, threshold = train quantile at expected rate
`4/24 ~ 0.1667` (the frozen Phase 5 policy form).

In [6]:
from backend.external_data.evaluation import external_rule_baseline

# The frozen rolling rule baseline is only applicable to an ordered series. It is
# therefore gated per dataset instead of applied blindly:
#   - bankruptcy: a cross-sectional test matrix with no temporal ordering -> UNAVAILABLE
#     (trailing-row windows over an ID-ordered matrix would be invalid).
#   - financial-statements: an annual panel -> evaluated independently PER COMPANY in
#     ascending year order using only past observations; company boundaries are never
#     crossed (groups/years come from the complete rows only, so they align 1:1 with
#     the feature frame the gate evaluates).
rows = []
for dataset_id, matrix in (("bankruptcy", bk_features), ("financial-statements", fs_features)):
    fitted, run, scores, flags = run_isolation_forest(matrix, dataset_id=dataset_id)
    if dataset_id == "financial-statements":
        keys = fs_raw.loc[matrix.dropna().index, ["Company", "Year"]]
        record = external_rule_baseline(
            matrix.dropna().reset_index(drop=True),
            dataset_id=dataset_id,
            groups=list(keys["Company"]),
            years=list(keys["Year"]),
        )
    else:
        record = external_rule_baseline(matrix.dropna().reset_index(drop=True), dataset_id=dataset_id)
    if record["available"]:
        rule_flagged = record["flagged_count"]
        rule_rate = record["flag_rate"]
        rule_note = f"scorable rows only ({record['n_scorable']}/{record['n_observations']})"
    else:
        rule_flagged = rule_rate = "unavailable"
        rule_note = record["reason"]
    rows.append(
        {
            "dataset": dataset_id,
            "n_modeled": run.n_observations,
            "n_features": run.n_features,
            "threshold": round(run.threshold, 6),
            "if_flagged": int(np.sum(flags)),
            "if_flag_rate": round(float(np.mean(flags)), 4),
            "rule_flagged": rule_flagged,
            "rule_flag_rate": rule_rate,
            "rule_basis": rule_note,
        }
    )
print(pd.DataFrame(rows).to_string(index=False))
print("\nNOTE: flag rates are near the expected rate BY CONSTRUCTION of the frozen threshold")
print("policy - they show the mechanism runs unchanged, not anomaly prevalence.")
print("NOTE: the rule baseline is a breach rate, gated by applicability - it is not an")
print("      anomaly rate and is not comparable across datasets.")


             dataset  n_modeled  n_features  threshold  if_flagged  if_flag_rate rule_flagged rule_flag_rate                                                                                                                                                                                                                                                                                                                                                                             rule_basis
          bankruptcy       5000          64  -0.103541         834        0.1668  unavailable    unavailable The frozen rolling z-score/IQR rule baseline computes trailing windows over row order and is only valid for an ordered temporal series. This dataset is cross-sectional (no time column; row order is an identifier order), so every trailing-window statistic would be meaningless. The baseline is therefore recorded as unavailable rather than reported with an invalid flag rate.
financial-statements        160 

In [7]:
# Label-gated evaluation: explicit unavailability records (nothing fabricated).
for dataset_id in ("bankruptcy", "financial-statements"):
    outcome = read_json(PROCESSED / dataset_id / "evaluation.json")
    print(f"{dataset_id}: available={outcome['available']}")
    print("  reason:", outcome["reason"][:160], "...")
assert read_json(PROCESSED / "bankruptcy" / "evaluation.json")["available"] is False
assert read_json(PROCESSED / "financial-statements" / "evaluation.json")["available"] is False
print(
    "\n=> No PR-AUC / ROC-AUC / F1 / confusion matrix exists "
    "for either dataset. None was computed."
)

bankruptcy: available=False
  reason: Bankruptcy.csv is the unlabeled competition test feature matrix (byte-identical to lupitaastari/bankruptcy 'bankruptcy_Test_X.csv'). Its companion labeled train ...
financial-statements: available=False
  reason: Financial Statements.csv contains no target column of any kind; unsupervised anomaly scores cannot be evaluated against outcomes. ...

=> No PR-AUC / ROC-AUC / F1 / confusion matrix exists for either dataset. None was computed.


## 5. Explainability - Phase 5 explanation path (seeded permutation-family Shapley estimator)Live demonstration on a small sample; the full recorded run (100 evenly spaced rows perdataset) is in `explanations.json`. The frozen internal label `shap_exact` is a knownmisnomer (deferred item D-1) and is preserved only because the Phase 5 golden fixturedepends on it; the authoritative runtime class is recorded in `explainer_runtime_class`.No TreeSHAP / TreeExplainer is used anywhere (see report section 13).

In [8]:
from backend.ml_engine.explain import attribute_row, global_summary

for dataset_id, matrix in (("bankruptcy", bk_features), ("financial-statements", fs_features)):
    complete = matrix.dropna().reset_index(drop=True)
    fitted, _, _, _ = run_isolation_forest(matrix, dataset_id=dataset_id)  # same seed -> same model
    idx = sorted({int(i) for i in np.linspace(0, len(complete) - 1, min(25, len(complete)))})
    attr_rows = [attribute_row(fitted, complete, i) for i in idx]
    ranking = global_summary(attr_rows, fitted.feature_names)
    top = ranking[:8] if isinstance(ranking, list) else list(ranking.items())[:8]
    names = [t[0] if isinstance(t, tuple) else t for t in top]
    print(f"{dataset_id}: explainer={attr_rows[0].explainer}, sample={len(attr_rows)} rows")
    print("  top features:", names)
    recorded = read_json(PROCESSED / dataset_id / "explanations.json")
    print(
        "  recorded sample size:",
        recorded["sample_rows"],
        "| explainers:",
        recorded["explainer_kinds"],
    )

E:\AI Business Risk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


bankruptcy: explainer=shap_exact, sample=25 rows
  top features: ['Attr50', 'Attr64', 'Attr3', 'Attr28', 'Attr8', 'Attr63', 'Attr38', 'Attr40']
  recorded sample size: 100 | explainers: ['shap_exact']


financial-statements: explainer=shap_exact, sample=25 rows
  top features: ['Net Income', 'Market Cap(in B USD)', 'EBITDA', 'Cash Flow from Financial Activities', 'Revenue', 'Gross Profit', 'Inflation Rate(in US)', 'Cash Flow from Operating']
  recorded sample size: 100 | explainers: ['shap_exact']


## 6. Risk Engine mapping — unmodified `QuantitativeRiskEngine.evaluate_company`

In [9]:
evaluation = evaluate_companies(fs_raw)
summary = availability_summary(evaluation)
print(f"metrics computable: {summary['metrics_computable']} / {summary['metrics_total']}")
dims_sorted = sorted(summary["dimension_scores"].items())
table = pd.DataFrame({dim: pd.Series(values) for dim, values in dims_sorted})
print("\nDimension scores by company (None = inputs unavailable, never fabricated):")
print(table.round(2).to_string())
print("\nComposite (available dimensions only):")
print(pd.Series(summary["composite_scores"]).round(2).to_string())
print(
    "\nBankruptcy dataset: 0/38 Risk Engine metrics computable "
    "(anonymized ratios, no semantics)."
)

metrics computable: 15 / 38

Dimension scores by company (None = inputs unavailable, never fabricated):
       concentration  credit  financial_strength liquidity  macro  market  operational
AAPL           18.33    15.0               23.73      None   35.0   33.75         12.5
AIG            18.33    15.0               18.40      None   35.0   33.75         12.5
AMZN           18.33    15.0               39.33      None   35.0   33.75         12.5
BCS            18.33    15.0               47.06      None   35.0   49.00         12.5
GOOG           18.33    15.0               16.36      None   35.0   31.93         12.5
INTC           18.33    15.0               26.28      None   35.0   33.75         12.5
MCD            18.33    15.0               47.90      None   35.0   33.14         12.5
MSFT           18.33    15.0               15.83      None   35.0   28.16         12.5
NVDA           18.33    15.0               19.08      None   35.0   33.75         12.5
PCG            18.33    15

## 7. Visualizations — figures generated by the experiment run

All PNGs below were produced by `python -m backend.external_data.run_experiment`
(DPI 200, deterministic, no fabricated curves). Label-gated slots are honest
"unavailable" notices.

In [10]:
from IPython.display import Markdown, display

figure_names = (
    "bankruptcy/feature-distributions.png",
    "bankruptcy/anomaly-score-distribution.png",
    "bankruptcy/shap-summary.png",
    "bankruptcy/pr-curve.png",
    "financial-statements/dataset-overview.png",
    "financial-statements/company-coverage.png",
    "financial-statements/financial-trends.png",
    "financial-statements/risk-dimension-distribution.png",
    "financial-statements/anomaly-results.png",
    "financial-statements/shap-summary.png",
    "cross-dataset/risk-dimension-comparison.png",
    "cross-dataset/feature-importance-comparison.png",
)

# Verify every figure exists on disk (real execution evidence), then link the
# committed PNGs by repository-relative path. The 17 PNGs are committed under
# docs/assets/external-evaluation/; they are NOT re-embedded as base64 here,
# which previously doubled the notebook size (~2.3 MB) with duplicated pixels.
for name in figure_names:
    path = FIGURES / name
    assert path.exists(), f"missing figure: {path}"
    rel = path.relative_to(REPO).as_posix()
    size_kb = path.stat().st_size // 1024
    nb_rel = (Path("..") / ".." / rel).as_posix()
    display(Markdown(f"**{rel}** ({size_kb} KB)  \n\n![]({nb_rel})"))


**docs/assets/external-evaluation/bankruptcy/feature-distributions.png** (174 KB)  

![](../../docs/assets/external-evaluation/bankruptcy/feature-distributions.png)

**docs/assets/external-evaluation/bankruptcy/anomaly-score-distribution.png** (60 KB)  

![](../../docs/assets/external-evaluation/bankruptcy/anomaly-score-distribution.png)

**docs/assets/external-evaluation/bankruptcy/shap-summary.png** (206 KB)  

![](../../docs/assets/external-evaluation/bankruptcy/shap-summary.png)

**docs/assets/external-evaluation/bankruptcy/pr-curve.png** (30 KB)  

![](../../docs/assets/external-evaluation/bankruptcy/pr-curve.png)

**docs/assets/external-evaluation/financial-statements/dataset-overview.png** (62 KB)  

![](../../docs/assets/external-evaluation/financial-statements/dataset-overview.png)

...

**docs/assets/external-evaluation/financial-statements/risk-dimension-distribution.png** (69 KB)  

![](../../docs/assets/external-evaluation/financial-statements/risk-dimension-distribution.png)

**docs/assets/external-evaluation/financial-statements/anomaly-results.png** (304 KB)  

![](../../docs/assets/external-evaluation/financial-statements/anomaly-results.png)

**docs/assets/external-evaluation/financial-statements/shap-summary.png** (252 KB)  

![](../../docs/assets/external-evaluation/financial-statements/shap-summary.png)

**docs/assets/external-evaluation/cross-dataset/risk-dimension-comparison.png** (46 KB)  

![](../../docs/assets/external-evaluation/cross-dataset/risk-dimension-comparison.png)

**docs/assets/external-evaluation/cross-dataset/feature-importance-comparison.png** (100 KB)  

![](../../docs/assets/external-evaluation/cross-dataset/feature-importance-comparison.png)

## 8. Cross-dataset summary (independent evaluation - never merged)

| | Bankruptcy | Financial Statements |
|---|---|---|
| Shape | 5,000 x 65 | 161 x 23 (12 cos, 2009-2023 annual) |
| Labels | NONE (recovery failed -> unsupervised-only) | NONE |
| IF flag rate | ~16.7 % (by threshold construction) | ~16.9 % (by threshold construction) |
| Rule baseline | UNAVAILABLE (cross-sectional: no temporal ordering) | per-company, scorable rows only |
| Risk Engine | 0/38 metrics | 15/38 metrics; liquidity = None |
| Attribution | Phase 5 path, 100-row sample | Phase 5 path, 100-row sample |

## Conclusions

1. **Operational generalization: verified.** The frozen Phase 5 pipeline and the unmodified
   Phase 4 Risk Engine execute deterministically (seed 42) on both independent datasets with
   zero methodology changes; raw hashes unchanged; leakage audits clean.
2. **Predictive generalization: NOT established.** Neither dataset carries genuine labels, so
   no supervised metric (PR-AUC, ROC-AUC, F1, confusion matrix) exists - and none was invented.
   The bankruptcy label download failed (no Kaggle credentials), triggering the mandated
   unsupervised-only fallback.
3. **Risk Engine compatibility: real but partial.** 15/38 metrics compute from real external
   fields; 23 remain `MISSING_INPUT`; the liquidity dimension honestly refuses to score.
4. **Baseline applicability is gated, not assumed.** The rolling rule baseline is refused for
   the cross-sectional bankruptcy matrix and applied per company on the annual panel, so its
   breach rate is never compared across datasets.
5. **Attribution terminology corrected.** `shap_exact` is a frozen misnomer; the runtime class
   is a seeded permutation-family Shapley estimator with verified additivity.
6. **What would close the gap:** an external dataset with *verified* outcome labels matched to
   these rows. Without ground truth, only pipeline portability - not accuracy - can be claimed.
